In [ ]:


import random
from datetime import datetime
from datetime import timedelta

from faker import Faker
from pydeequ.analyzers import *

In [ ]:
spark = SparkSession.builder.appName("Example S3").getOrCreate()

In [ ]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

In [ ]:
def random_date(days=30):
    return datetime.now() - timedelta(days=random.randint(0, days))


def fake_natural_product_name():
    pattern = random.choice([
        lambda: fake.catch_phrase(),
        lambda: fake.bs().title(),
        lambda: f"{fake.color_name()} {fake.word().title()} Edition",
        lambda: f"{fake.word().title()} {fake.word().title()} Series",
        lambda: fake.sentence(nb_words=random.randint(3, 6)).replace(".", "")
    ])
    return pattern()


categories = ["전자기기", "가전", "패션", "도서", "생활용품"]
order_status = ["주문완료", "배송중", "배송완료", "취소"]
payment_method = ["카드", "계좌이체", "카카오페이", "네이버페이"]
orders = []
for i in range(30000):
    quantity = random.randint(1, 5)
    unit_price = random.randint(10000, 500000)
    orders.append({
        "order_id": i + 1,
        "user_id": random.randint(1, 100),  # 기존 user id
        "product_name": fake_natural_product_name(),
        "category": random.choice(categories),
        "quantity": quantity,
        "unit_price": unit_price,
        "total_price": quantity * unit_price,
        "order_status": random.choice(order_status),
        "order_date": random_date(),
        "payment_method": random.choice(payment_method),
        "shipping_address": fake.address()
    })

In [ ]:
dataframe = spark.createDataFrame(data=orders)